# Gemini-3-Flash-Preview

In [4]:
# Cell 1: Setup imports, paths, and Gemini key (new google.genai SDK)
from pathlib import Path
import os
import re
import json
import time
import subprocess
from datetime import datetime

from dotenv import load_dotenv
from google import genai

# Project root (run notebook from formalizer-scaling/src/pddl_gen)
PROJECT_ROOT = Path.cwd().parents[1]

DATA_DIR = PROJECT_ROOT / "data"
PROMPT_PATH = PROJECT_ROOT / "prompts" / "only_pddl_instruction.txt"
DOMAIN_PDDL_PATH = DATA_DIR / "pddl" / "domain.pddl"
DESC_DIR = DATA_DIR / "descriptions"

OUTPUT_ROOT = PROJECT_ROOT / "output" / "direct_pddl" / "gemini_3_flash"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

VAL_PATH = Path("/home/sagemaker-user/tools/VAL/build/linux64/Release/bin/Validate")

# Load .env and configure Gemini client
load_dotenv(PROJECT_ROOT / ".env")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
  raise ValueError("GOOGLE_API_KEY not found in .env")

client = genai.Client(api_key=GOOGLE_API_KEY)

# Gemini 3 Flash model code (preview)
MODEL_NAME = "gemini-3-flash-preview"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("VAL exists:", VAL_PATH.exists())
print("Domain PDDL exists:", DOMAIN_PDDL_PATH.exists())
print("Prompt exists:", PROMPT_PATH.exists())
print("Model:", MODEL_NAME)

PROJECT_ROOT: /home/sagemaker-user/formalizer-scaling
OUTPUT_ROOT: /home/sagemaker-user/formalizer-scaling/output/direct_pddl/gemini_3_flash
VAL exists: True
Domain PDDL exists: True
Prompt exists: True
Model: gemini-3-flash-preview


In [5]:
# Cell 2: Define target problems and load static prompt/domain text
PROBLEM_IDS = [f"p{i:03d}" for i in range(101, 201)]  # p101..p200

PROMPT_PREFIX = PROMPT_PATH.read_text()
DOMAIN_PDDL_TEXT = DOMAIN_PDDL_PATH.read_text()

print("Num problems:", len(PROBLEM_IDS))
print("First/last:", PROBLEM_IDS[0], PROBLEM_IDS[-1])
print("Prompt chars:", len(PROMPT_PREFIX))
print("Domain PDDL chars:", len(DOMAIN_PDDL_TEXT))

Num problems: 100
First/last: p101 p200
Prompt chars: 2544
Domain PDDL chars: 997


In [6]:
# Cell 3: Load NL descriptions for one problem ID
def load_problem_descriptions(problem_id: str):
  domain_path = DESC_DIR / f"{problem_id}_domain.txt"
  problem_path = DESC_DIR / f"{problem_id}_problem.txt"

  if not domain_path.exists():
      raise FileNotFoundError(f"Missing domain description: {domain_path}")
  if not problem_path.exists():
      raise FileNotFoundError(f"Missing problem description: {problem_path}")

  return domain_path.read_text(), problem_path.read_text()

In [7]:
# Cell 4: Build prompt for one problem (combined primer + 3-input setup)
def build_prompt(problem_id: str):
  nl_domain, nl_problem = load_problem_descriptions(problem_id)

  return (
      f"{PROMPT_PREFIX}\n\n"
      "You are an expert PDDL engineer.\n"
      "Create a syntactically correct and semantically consistent problem file "
      "using the information below.\n\n"
      "You are given three inputs:\n"
      "1) Original natural language domain description\n"
      "2) Original natural language problem description\n"
      "3) Ground-truth domain PDDL (fixed reference)\n\n"
      "### Original domain description\n"
      f"{nl_domain}\n\n"
      "### Original problem description\n"
      f"{nl_problem}\n\n"
      "### Ground-truth domain PDDL\n"
      f"{DOMAIN_PDDL_TEXT}\n\n"
      "Output contract:\n"
      "- Generate ONLY the final problem PDDL.\n"
      "- Wrap it in exactly one <problem_file>...</problem_file> block.\n"
      "- Do not output any <domain_file> block.\n"
      "- Do not include any extra text before or after <problem_file>...</problem_file>."
  )

In [9]:
# Cell 5: Call Gemini for one prompt
def call_gemini(prompt: str, temperature: float = 0.4, max_output_tokens: int = 10000):
  response = client.models.generate_content(
      model=MODEL_NAME,
      contents=prompt,
      config={
          "temperature": temperature,
          "max_output_tokens": max_output_tokens,
      },
  )
  return (response.text or "").strip()

In [10]:
# Cell 6: Parse model output and extract problem_file block
def extract_problem_pddl(raw_text: str):
  if not raw_text:
      return ""

  matches = re.findall(r"<problem_file>(.*?)</problem_file>", raw_text, flags=re.DOTALL | re.IGNORECASE)
  if matches:
      return matches[-1].strip()

  return ""

In [11]:
# Cell 7: Solve one generated problem with live logs (planutils dual-bfws-ffparser)
import select

def run_solver(domain_pddl_path: Path, problem_pddl_path: Path, work_dir: Path, timeout_sec: int = 300):
  plan_path = work_dir / "plan"
  if plan_path.exists():
      plan_path.unlink()

  cmd = ["planutils", "run", "dual-bfws-ffparser", str(domain_pddl_path), str(problem_pddl_path)]
  print(f"[solver] start: {' '.join(cmd)}")
  print(f"[solver] timeout: {timeout_sec}s")

  proc = subprocess.Popen(
      cmd,
      cwd=work_dir,
      stdout=subprocess.PIPE,
      stderr=subprocess.PIPE,
      text=True,
      bufsize=1,
  )

  stdout_lines = []
  stderr_lines = []
  start = time.time()

  # Live-stream both stdout and stderr while process runs
  while True:
      if proc.poll() is not None:
          break

      elapsed = time.time() - start
      if elapsed > timeout_sec:
          proc.kill()
          print(f"[solver] timeout reached ({timeout_sec}s), process killed.")
          break

      ready, _, _ = select.select([proc.stdout, proc.stderr], [], [], 0.2)
      for stream in ready:
          line = stream.readline()
          if not line:
              continue
          line = line.rstrip("\n")
          if stream is proc.stdout:
              stdout_lines.append(line)
              print(f"[solver][out] {line}")
          else:
              stderr_lines.append(line)
              print(f"[solver][err] {line}")

  # Drain remaining output
  out_rest, err_rest = proc.communicate(timeout=2)
  if out_rest:
      for line in out_rest.splitlines():
          stdout_lines.append(line)
          print(f"[solver][out] {line}")
  if err_rest:
      for line in err_rest.splitlines():
          stderr_lines.append(line)
          print(f"[solver][err] {line}")

  stdout = "\n".join(stdout_lines).strip()
  stderr = "\n".join(stderr_lines).strip()

  if plan_path.exists():
      plan_text = plan_path.read_text().strip()
      if plan_text:
          print("[solver] plan found.")
          return {
              "plan_found": True,
              "plan_text": plan_text,
              "solver_error": "",
              "stdout": stdout,
              "stderr": stderr,
          }

  if (time.time() - start) > timeout_sec and proc.returncode is None:
      solver_error = f"timeout after {timeout_sec}s"
  else:
      solver_error = (stderr + "\n" + stdout).strip() or "plan not found"

  print("[solver] no valid plan file produced.")
  return {
      "plan_found": False,
      "plan_text": "",
      "solver_error": solver_error,
      "stdout": stdout,
      "stderr": stderr,
  }
